# Open-Source-Python-Package — Colab-safe version

# Open-Source-Python-Package

This notebook introduces a lightweight **open-source Python package** for NLP-oriented experimentation, prototyping, and reproducible research sharing in Google Colab.

Its design is centered around five complementary directions:
- **sentiment analysis**
- **text preprocessing**
- **keyword extraction**
- **stream-style chunk summaries**
- **reproducible packaging for GitHub**

The notebook provides a compact and stable framework for modular text analytics, lightweight lexical exploration, chunk-level summary generation, and reproducible packaging practices aligned with public research code dissemination.

# Stable installation for Colab

In [ ]:
import subprocess, sys

def run(cmd):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

print("Python:", sys.version)

run("pip -q uninstall -y gradio gradio_client huggingface_hub fastapi starlette pydantic pydantic_core >/dev/null 2>&1 || true")
run(
    "pip -q install --no-cache-dir "
    "'huggingface_hub==0.24.6' "
    "'fastapi==0.112.2' "
    "'starlette==0.38.2' "
    "'pydantic==2.8.2' "
    "'gradio==4.31.5' "
    "'gradio_client==0.16.4' "
    "'tabulate==0.9.0'"
)

print("\nInstallation completed.")
print("If this Colab runtime had old packages loaded, do: Runtime > Restart runtime, then Run all.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
>> pip -q uninstall -y gradio gradio_client huggingface_hub fastapi starlette pydantic pydantic_core >/dev/null 2>&1 || true
>> pip -q install --no-cache-dir 'huggingface_hub==0.24.6' 'fastapi==0.112.2' 'starlette==0.38.2' 'pydantic==2.8.2' 'gradio==4.31.5' 'gradio_client==0.16.4' 'tabulate==0.9.0'

Installation completed.
If this Colab runtime had old packages loaded, do: Runtime > Restart runtime, then Run all.


# Imports and configuration

In [ ]:
import csv
import os
import re
import sys
import zipfile
from collections import Counter

import gradio as gr
from tabulate import tabulate

SEED = 42

POSITIVE_WORDS = {
    "good","great","excellent","amazing","love","loved","like","liked","helpful","fast","smooth","awesome",
    "fantastic","positive","satisfied","clean","efficient","useful","improved","impressive","nice","best",
    "reliable","stable","friendly","quick","support","accurate","success","effective","clear","robust",
    "smart","simple","easy","powerful","secure","responsive","productive"
}

NEGATIVE_WORDS = {
    "bad","terrible","awful","hate","hated","slow","bug","bugs","frustrating","poor","negative","disappointed",
    "confusing","worse","worst","unreliable","broken","issue","issues","delay","late","hard","difficult",
    "problem","problems","error","errors","unstable","boring","noisy","weak","fail","failed","failure",
    "crash","crashes","wrong","ugly","annoying","risky"
}

STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","with","is","it","this","that","was","were","be",
    "as","at","by","from","are","but","if","then","than","so","we","you","they","he","she","i","my","our",
    "their","them","his","her","its","about","into","after","before","during","can","could","would","should",
    "will","do","does","did","have","has","had","not","very","more","most","much","many","just","also"
}

print("Python:", sys.version.split()[0])
print("Gradio:", gr.__version__)
print("Imports loaded successfully.")

Python: 3.12.13
Gradio: 4.31.5
Imports loaded successfully.


# Package source code generator

In [ ]:
PACKAGE_NAME = "haddadnlpkit"
ROOT_DIR = "/content/haddadnlpkit_project"
PKG_DIR = os.path.join(ROOT_DIR, PACKAGE_NAME)

os.makedirs(PKG_DIR, exist_ok=True)

INIT_CODE = '''
"""haddadnlpkit: lightweight NLP utilities for Colab-safe demos."""

from .core import (
    clean_text,
    tokenize,
    extract_keywords,
    lexicon_sentiment,
    hybrid_sentiment,
    stream_summary,
    batch_from_rows,
)
'''

CORE_CODE = r'''
import re
from collections import Counter

POSITIVE_WORDS = {
    "good","great","excellent","amazing","love","loved","like","liked","helpful","fast","smooth","awesome",
    "fantastic","positive","satisfied","clean","efficient","useful","improved","impressive","nice","best",
    "reliable","stable","friendly","quick","support","accurate","success","effective","clear","robust",
    "smart","simple","easy","powerful","secure","responsive","productive"
}

NEGATIVE_WORDS = {
    "bad","terrible","awful","hate","hated","slow","bug","bugs","frustrating","poor","negative","disappointed",
    "confusing","worse","worst","unreliable","broken","issue","issues","delay","late","hard","difficult",
    "problem","problems","error","errors","unstable","boring","noisy","weak","fail","failed","failure",
    "crash","crashes","wrong","ugly","annoying","risky"
}

STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","with","is","it","this","that","was","were","be",
    "as","at","by","from","are","but","if","then","than","so","we","you","they","he","she","i","my","our",
    "their","them","his","her","its","about","into","after","before","during","can","could","would","should",
    "will","do","does","did","have","has","had","not","very","more","most","much","many","just","also"
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    text = clean_text(text)
    return [tok for tok in text.split() if tok and tok not in STOPWORDS]

def extract_keywords(text, top_k=10):
    tokens = tokenize(text)
    counts = Counter(tokens).most_common(int(top_k))
    return [{"token": token, "count": count} for token, count in counts]

def lexicon_sentiment(text):
    tokens = tokenize(text)
    if not tokens:
        return {"negative": 0.0, "positive": 0.0, "neutral": 1.0, "label": "neutral", "confidence": 0.0}
    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    total_hits = pos + neg
    if total_hits == 0:
        return {"negative": 0.15, "positive": 0.15, "neutral": 0.70, "label": "neutral", "confidence": 0.70}
    positive = pos / total_hits
    negative = neg / total_hits
    neutral = max(0.0, 1.0 - max(positive, negative))
    label = "positive" if positive > negative else "negative" if negative > positive else "neutral"
    confidence = max(positive, negative, neutral)
    return {"negative": round(negative, 4), "positive": round(positive, 4), "neutral": round(neutral, 4), "label": label, "confidence": round(confidence, 4)}

def hybrid_sentiment(text, alpha=0.65):
    tokens = tokenize(text)
    base = lexicon_sentiment(text)
    if not tokens:
        return base
    density = min(len(tokens) / 30.0, 1.0)
    refined_pos = alpha * base["positive"] + (1.0 - alpha) * (density * 0.5 + base["positive"] * 0.5)
    refined_neg = alpha * base["negative"] + (1.0 - alpha) * (density * 0.2 + base["negative"] * 0.8)
    s = refined_pos + refined_neg
    neutral = max(0.0, 1.0 - s)
    if refined_pos >= refined_neg and refined_pos >= neutral:
        label = "positive"
        confidence = refined_pos
    elif refined_neg >= refined_pos and refined_neg >= neutral:
        label = "negative"
        confidence = refined_neg
    else:
        label = "neutral"
        confidence = neutral
    return {
        "negative": round(refined_neg, 4),
        "positive": round(refined_pos, 4),
        "neutral": round(neutral, 4),
        "label": label,
        "confidence": round(confidence, 4),
    }

def stream_summary(texts, chunk_size=5, alpha=0.65):
    chunk_size = max(1, int(chunk_size))
    rows = []
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i:i + chunk_size]
        labels = []
        confidences = []
        lengths = []
        for text in chunk:
            s = hybrid_sentiment(text, alpha=alpha)
            labels.append(s["label"])
            confidences.append(s["confidence"])
            lengths.append(len(str(text)))
        dominant = Counter(labels).most_common(1)[0][0] if labels else "neutral"
        avg_conf = round(sum(confidences) / len(confidences), 4) if confidences else 0.0
        avg_len = round(sum(lengths) / len(lengths), 2) if lengths else 0.0
        rows.append({
            "chunk_id": len(rows) + 1,
            "chunk_size": len(chunk),
            "dominant_label": dominant,
            "avg_confidence": avg_conf,
            "avg_length_chars": avg_len,
        })
    return rows

def batch_from_rows(rows, text_key="text", alpha=0.65):
    out = []
    for row in rows:
        text = str(row.get(text_key, ""))
        s = hybrid_sentiment(text, alpha=alpha)
        new_row = dict(row)
        new_row.update({
            "predicted_label": s["label"],
            "confidence": s["confidence"],
            "negative": s["negative"],
            "neutral": s["neutral"],
            "positive": s["positive"],
        })
        out.append(new_row)
    return out
'''

README_CODE = '''
# haddadnlpkit

A lightweight open-source Python package for Colab-safe NLP demos.

## Features
- text cleaning
- tokenization
- keyword extraction
- hybrid lexicon-style sentiment analysis
- stream-style chunk summaries
- batch processing from CSV/TXT derived rows

## Install
```bash
pip install .
```

## Example
```python
from haddadnlpkit import hybrid_sentiment, extract_keywords

print(hybrid_sentiment("The interface is simple, fast, and very useful."))
print(extract_keywords("The interface is simple, fast, and very useful.", top_k=5))
```
'''

PYPROJECT = '''
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "haddadnlpkit"
version = "0.1.0"
description = "Lightweight NLP utilities for Colab-safe demos"
readme = "README.md"
requires-python = ">=3.10"
authors = [
    {name = "Omar Haddad"}
]
'''

SETUP_CFG = '''
[options]
packages = find:
include_package_data = True
zip_safe = False
'''

with open(os.path.join(PKG_DIR, "__init__.py"), "w", encoding="utf-8") as f:
    f.write(INIT_CODE.strip() + "\n")

with open(os.path.join(PKG_DIR, "core.py"), "w", encoding="utf-8") as f:
    f.write(CORE_CODE.strip() + "\n")

with open(os.path.join(ROOT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(README_CODE.strip() + "\n")

with open(os.path.join(ROOT_DIR, "pyproject.toml"), "w", encoding="utf-8") as f:
    f.write(PYPROJECT.strip() + "\n")

with open(os.path.join(ROOT_DIR, "setup.cfg"), "w", encoding="utf-8") as f:
    f.write(SETUP_CFG.strip() + "\n")

print("Package source files created in:", ROOT_DIR)
print(os.listdir(ROOT_DIR))

Package source files created in: /content/haddadnlpkit_project
['build', 'README.md', 'haddadnlpkit', 'pyproject.toml', 'haddadnlpkit.egg-info', 'setup.cfg']


# Install the local package

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", ROOT_DIR])

from haddadnlpkit import (
    clean_text,
    tokenize,
    extract_keywords,
    lexicon_sentiment,
    hybrid_sentiment,
    stream_summary,
    batch_from_rows,
)

print("Local package installed and imported successfully.")
print(hybrid_sentiment("The package is simple, robust, and useful."))

Local package installed and imported successfully.
{'negative': 0.0093, 'positive': 0.8483, 'neutral': 0.1423, 'label': 'positive', 'confidence': 0.8483}


# Helpers for file loading and Markdown rendering

In [ ]:
def rows_to_markdown(rows, max_rows=20):
    if not rows:
        return "No data available."
    display_rows = rows[:max_rows]
    headers = list(display_rows[0].keys())
    table = [[row.get(h, "") for h in headers] for row in display_rows]
    return tabulate(table, headers=headers, tablefmt="github")

def dict_to_markdown(d):
    lines = ["### Summary"]
    for k, v in d.items():
        lines.append(f"- **{k}**: {v}")
    return "\n".join(lines)

def parse_txt_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append({"text": line})
    return rows

def parse_csv_file(path, text_column="text"):
    with open(path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return []
    columns = list(rows[0].keys())
    if text_column not in columns:
        candidates = [c for c in columns if c]
        if not candidates:
            return []
        text_column = candidates[0]
    normalized = []
    for row in rows:
        normalized.append({"text": str(row.get(text_column, "")), "source_column": text_column})
    return normalized

def load_rows(file_path=None, text_column="text"):
    if file_path is None:
        return [
            {"text": "I loved the quality of this product and the support team was amazing."},
            {"text": "This update is terrible, slow, and full of frustrating bugs."},
            {"text": "The service is okay, but the response time could be better."},
            {"text": "A clean and effective workflow with reliable results."},
            {"text": "The interface is confusing and the app crashed twice."},
            {"text": "A robust and simple package for reproducible text processing."},
        ]
    lower = file_path.lower()
    if lower.endswith(".txt"):
        return parse_txt_file(file_path)
    if lower.endswith(".csv"):
        return parse_csv_file(file_path, text_column=text_column)
    raise ValueError("Unsupported file format. Use CSV or TXT only.")

print("Helper functions ready.")

Helper functions ready.


# Core app functions

In [ ]:
def analyze_single_text(text, alpha):
    text = str(text or "")
    result = hybrid_sentiment(text, alpha=float(alpha))
    keywords = extract_keywords(text, top_k=12)
    stream = stream_summary([text], chunk_size=1, alpha=float(alpha))
    summary_md = dict_to_markdown(result)
    keywords_md = "### Top keywords\n\n" + rows_to_markdown(keywords, max_rows=12)
    stream_md = "### Stream-style summary\n\n" + rows_to_markdown(stream, max_rows=5)
    cleaned = clean_text(text)
    preview = cleaned[:500] + ("..." if len(cleaned) > 500 else "")
    clean_md = "### Cleaned text\n\n```\n" + preview + "\n```"
    return summary_md, keywords_md, stream_md, clean_md

def analyze_batch(file_path, text_column, alpha, chunk_size):
    rows = load_rows(file_path=file_path, text_column=text_column)
    batch_rows = batch_from_rows(rows, text_key="text", alpha=float(alpha))
    preview_md = "### Batch predictions\n\n" + rows_to_markdown(batch_rows, max_rows=25)
    stream_rows = stream_summary([r["text"] for r in rows], chunk_size=int(chunk_size), alpha=float(alpha))
    stream_md = "### Batch vs stream-style chunk summary\n\n" + rows_to_markdown(stream_rows, max_rows=25)

    label_counts = Counter(r["predicted_label"] for r in batch_rows)
    stats_rows = [{"label": k, "count": v} for k, v in sorted(label_counts.items())]
    stats_md = "### Label distribution\n\n" + rows_to_markdown(stats_rows, max_rows=10)

    output_csv = "/content/haddadnlpkit_batch_predictions.csv"
    fieldnames = list(batch_rows[0].keys()) if batch_rows else ["text", "predicted_label", "confidence", "negative", "neutral", "positive"]
    with open(output_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in batch_rows:
            writer.writerow(row)

    return preview_md, stream_md, stats_md, output_csv

def build_project_archive():
    archive_path = "/content/haddadnlpkit_project.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for base, _, files in os.walk(ROOT_DIR):
            for name in files:
                full = os.path.join(base, name)
                arc = os.path.relpath(full, ROOT_DIR)
                zf.write(full, arcname=arc)
    return archive_path

print("App functions ready.")

App functions ready.


# Dry run tests

In [ ]:
print("Single text test:")
single_out = analyze_single_text(
    "The package is simple, effective, and robust, but one screen is a bit confusing.",
    0.65
)
print(single_out[0])
print()

print("Batch test:")
batch_out = analyze_batch(None, "text", 0.65, 3)
print(batch_out[2])
print("CSV export path:", batch_out[3])

archive = build_project_archive()
print("Project archive:", archive)
print("Dry run completed successfully.")

Single text test:
### Summary
- **negative**: 0.2512
- **positive**: 0.6654
- **neutral**: 0.0834
- **label**: positive
- **confidence**: 0.6654

Batch test:
### Label distribution

| label    |   count |
|----------|---------|
| negative |       2 |
| neutral  |       1 |
| positive |       3 |
CSV export path: /content/haddadnlpkit_batch_predictions.csv
Project archive: /content/haddadnlpkit_project.zip
Dry run completed successfully.


# Gradio application

In [ ]:
examples = [
    ["I loved the quality of this product and the support team was amazing."],
    ["This update is terrible, slow, and full of frustrating bugs."],
    ["The service is okay, but the response time could be better."],
]

with gr.Blocks(title="Open-Source-Python-Package") as demo:
    gr.Markdown("# Open-Source-Python-Package")
    gr.Markdown(
        "A Colab-safe open-source package demo for practical NLP analysis. This application focuses on packaging, sentiment analysis, keyword extraction, batch inference, and stream-style chunk summaries."
    )

    with gr.Tab("Single text"):
        text_in = gr.Textbox(lines=8, label="Input text")
        alpha_in = gr.Slider(0.0, 1.0, value=0.65, step=0.05, label="Hybrid weight (alpha)")
        run_single = gr.Button("Analyze")
        summary_out = gr.Markdown()
        keywords_out = gr.Markdown()
        stream_out = gr.Markdown()
        clean_out = gr.Markdown()

        gr.Examples(examples=examples, inputs=[text_in])

        run_single.click(
            fn=analyze_single_text,
            inputs=[text_in, alpha_in],
            outputs=[summary_out, keywords_out, stream_out, clean_out],
            api_name=False
        )

    with gr.Tab("Batch CSV/TXT"):
        file_in = gr.File(label="Upload CSV or TXT", file_types=[".csv", ".txt"], type="filepath")
        text_col_in = gr.Textbox(label="Text column for CSV", value="text")
        alpha_batch = gr.Slider(0.0, 1.0, value=0.65, step=0.05, label="Hybrid weight (alpha)")
        chunk_in = gr.Slider(1, 20, value=5, step=1, label="Chunk size")
        run_batch = gr.Button("Run batch analysis")

        batch_preview = gr.Markdown()
        batch_stream = gr.Markdown()
        batch_stats = gr.Markdown()
        batch_file = gr.File(label="Download batch predictions")

        run_batch.click(
            fn=analyze_batch,
            inputs=[file_in, text_col_in, alpha_batch, chunk_in],
            outputs=[batch_preview, batch_stream, batch_stats, batch_file],
            api_name=False
        )

    with gr.Tab("Package export"):
        gr.Markdown(
            "This tab exports the package source tree so you can place it directly into a GitHub repository."
        )
        export_btn = gr.Button("Build project ZIP")
        export_file = gr.File(label="Download project ZIP")
        export_btn.click(
            fn=build_project_archive,
            inputs=[],
            outputs=[export_file],
            api_name=False
        )

demo.queue()
demo.launch(share=True, debug=False, show_api=False)

IMPORTANT: You are using gradio version 4.31.5, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://3d2d37e30d7dace024.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
